In [1]:
import pandas as pd
import numpy as np
np.random.seed(94357)

# Primer conjunto de datos (nodos 0..20000)
Este conjunto corresponde al subgrafo construido con los nodos identificados del 0 al 20000 (rango fijo y reproducible).
Se usa como muestra manejable del grafo completo para extraer relaciones y combinar atributos.
El objetivo es reducir el volumen sin perder estructura, facilitando el analisis y el modelado.

## Primer modelo: Naive Bayes

Aplicaremos Naive Bayes como linea base porque es rapido y funciona bien con variables independientes o casi independientes. El modelo estima la probabilidad de cada clase dado un conjunto de caracteristicas y elige la clase con mayor probabilidad. Aunque la suposicion de independencia es fuerte, suele dar resultados competitivos y nos sirve para comparar con modelos mas complejos.

In [19]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

In [68]:
from sklearn.preprocessing import KBinsDiscretizer

# Cargar datos y preparar X/y
ruta = "generated_CSV/features_relaciones_20k.csv"

df = pd.read_csv(ruta)
columnas_excluir = ["created_at", "update_at", "numeric_id"]
columnas_relacionales = ["degree_centrality", "closeness_centrality", "betweenness_centrality", "clustering", "pageRank"]

y = df["affiliate"].copy()
#DF 1: Todos los atributos excepto los de texto y la variable objetivo
X = df.drop(columns=columnas_excluir + ["affiliate"], errors="ignore").copy()
#DF 2: No atributos relacionales
#X = df.drop(columns=columnas_excluir + ["affiliate"] + columnas_relacionales, errors="ignore").copy()
#DF 3: Solo atributos relacionales
# X = df[columnas_relacionales].copy()
#DF 4: Todos los atributos originiales pero añadiendo solo pageRank y closeness_centrality
#X = df.drop(columns=columnas_excluir + ["affiliate"], errors="ignore").copy()
#X["pageRank"] = df["pageRank"]
# X["closeness_centrality"] = df["closeness_centrality"]
#DF 5: Atributos seleccionados por chi2
#columnas_seleccionadas = ["views", "life_time", "betweenness_centrality"]
#X = df[columnas_seleccionadas].copy()

# Discretizar atributos continuos antes de codificar
atributos_continuos = ["degree_centrality", "closeness_centrality", "betweenness_centrality", "clustering", "pageRank"]

discretizador = KBinsDiscretizer(
    n_bins=40,
    encode="ordinal",
    strategy="kmeans",
)
X_disc = X.copy()
#Con el DF 2 hay que comentar esta parte porque no tenemos atributos continuos
atr_contenidos = list(set(atributos_continuos) & set(X_disc.columns))
X_disc[atr_contenidos] = discretizador.fit_transform(
    X_disc[atr_contenidos]
 )

# Codificar variables categoricas (incluye numericas como categorias si vienen como object)
encoder = OrdinalEncoder()
X_enc = encoder.fit_transform(X_disc)

# Codificar la variable objetivo
label_enc = LabelEncoder()
y_enc = label_enc.fit_transform(y)

In [69]:
# Particion train/test
atr_train, atr_test, obj_train, obj_test = train_test_split(
    X_enc, y_enc, test_size=0.2, random_state=94357, stratify=y_enc
)

In [70]:
tubería_NB = Pipeline([('preprocesador', discretizador),
                       ('naive_Bayes', CategoricalNB())])
rejilla_de_hiperparámetros = {
    'naive_Bayes__alpha': range(1, 10),
    'naive_Bayes__force_alpha': [True, False],
}


In [71]:
búsqueda_en_rejilla = GridSearchCV(tubería_NB,
                                   rejilla_de_hiperparámetros,
                                   scoring='accuracy',
                                   cv=10)
búsqueda_en_rejilla.fit(atr_train, obj_train)

c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (40). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 1 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (40). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in 

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...goricalNB())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'naive_Bayes__alpha': range(1, 10), 'naive_Bayes__force_alpha': [True, False]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",10
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the 

In [72]:
búsqueda_en_rejilla.best_estimator_

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocesador', ...), ('naive_Bayes', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"n_bins n_bins: int or array-like of shape (n_features,), default=5The number of bins to produce. Raises ValueError if ``n_bins < 2``.",40
,"encode encode: {'onehot', 'onehot-dense', 'ordinal'}, default='onehot'Method used to encode the transformed result.- 'onehot': Encode the transformed result with one-hot encoding and return a sparse matrix. Ignored features are always stacked to the right.- 'onehot-dense': Encode the transformed result with one-hot encoding and return a dense array. Ignored features are always stacked to the right.- 'ordinal': Return the bin identifier encoded as an integer value.",'ordinal'
,"strategy strategy: {'uniform', 'quantile', 'kmeans'}, default='quantile'Strategy used to define the widths of the bins.- 'uniform': All bins in each feature have identical widths.- 'quantile': All bins in each feature have the same number of points.- 'kmeans': Values in each bin have the same nearest center of a 1D k-means cluster.For an example of the different strategies see::ref:`sphx_glr_auto_examples_preprocessing_plot_discretization_strategies.py`.",'kmeans'
,"quantile_method quantile_method: {""inverted_cdf"", ""averaged_inverted_cdf"",""closest_observation"", ""interpolated_inverted_cdf"", ""hazen"",""weibull"", ""linear"", ""median_unbiased"", ""normal_unbiased""},default=""linear""Method to pass on to np.percentile calculation when usingstrategy=""quantile"". Only `averaged_inverted_cdf` and `inverted_cdf`support the use of `sample_weight != None` when subsampling is notactive... versionadded:: 1.7",'warn'
,"dtype dtype: {np.float32, np.float64}, default=NoneThe desired data-type for the output. If None, output dtype isconsistent with input dtype. Only np.float32 and np.float64 aresupported... versionadded:: 0.24",None
,"subsample subsample: int or None, default=200_000Maximum number of samples, used to fit the model, for computationalefficiency.`subsample=None` means that all the training samples are used whencomputing the quantiles that determine the binning thresholds.Since quantile computation relies on sorting each column of `X` andthat sorting has an `n log(n)` time complexity,it is recommended to u

In [73]:
búsqueda_en_rejilla.best_score_

np.float64(0.7460073931696821)

In [74]:
# Evaluacion
tubería_NB = Pipeline([('preprocesador', discretizador),
                       ('naive_Bayes', CategoricalNB(alpha=2, force_alpha=True))])
tubería_NB.fit(atr_train, obj_train)
tubería_NB.score(atr_test, obj_test)

c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (40). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 1 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (40). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in 

0.7421643997634536

In [75]:
predicciones_NB = tubería_NB.predict(atr_test)
print(classification_report(obj_test, predicciones_NB, target_names=[str(c) for c in label_enc.classes_]))

              precision    recall  f1-score   support

           0       0.76      0.67      0.71      1611
           1       0.73      0.81      0.77      1771

    accuracy                           0.74      3382
   macro avg       0.74      0.74      0.74      3382
weighted avg       0.74      0.74      0.74      3382



In [ ]:
#OBTENCIÓN DE LAS MEJORES FEATURES CON CHI2
from sklearn.feature_selection import SelectKBest, chi2

X_cat = X_enc.astype(int)

chi2_features = SelectKBest(chi2, k=3)
X_kbest_features = chi2_features.fit_transform(X_cat, y_enc)

# Variables reducidas
print("Original feature number:", X_cat.shape[1])
print("Reduced feature number:", X_kbest_features.shape[1])


feature_names = X.columns.tolist()
selected_mask = chi2_features.get_support() # Devuelve un array booleano indicando qué columnas han sido seleccionadas
selected_features = [name for name, keep in zip(feature_names, selected_mask) if keep]
print("Selected features:", selected_features)

## Segundo modelo: CART (Classification and Regression Trees)

CART es un algoritmo de aprendizaje supervisado que construye un árbol de decisión mediante divisiones binarias recursivas. En cada nodo, el algoritmo selecciona el atributo y el umbral que mejor separan las clases, maximizando la pureza de los subconjuntos resultantes (usando métricas como el índice Gini o la entropía).

A diferencia de Naive Bayes, CART no asume independencia entre las características y puede capturar relaciones no lineales e interacciones complejas entre variables. Además, es interpretable visualmente, ya que el árbol resultante muestra explícitamente las reglas de decisión.

Para evitar el sobreajuste, aplicaremos poda del árbol limitando su profundidad máxima (`max_depth`) y el número mínimo de muestras necesarias para dividir un nodo (`min_samples_split`). Estos hiperparámetros se optimizarán mediante búsqueda en rejilla con validación cruzada.

In [65]:
from sklearn.tree import DecisionTreeClassifier

In [66]:
clasificador_CART = DecisionTreeClassifier(random_state=94357)
rejilla_de_hiperparámetros_CART = {
    'max_depth': range(5, 50), # profundidad máxima del árbol para evitar el overfitting
    'min_samples_split': range(5, 25, 5) # número de muestras mínimo para dividir un nodo y prevenir el overfitting
}

In [67]:
búsqueda_en_rejilla_CART = GridSearchCV(clasificador_CART,
                                        rejilla_de_hiperparámetros_CART,
                                        scoring='accuracy',
                                        cv=10)
búsqueda_en_rejilla_CART.fit(atr_train, obj_train)

KeyboardInterrupt: 

In [ ]:
búsqueda_en_rejilla_CART.best_params_

{'max_depth': 6, 'min_samples_split': 20}

In [ ]:
búsqueda_en_rejilla_CART.best_score_

np.float64(0.7701062617807458)

In [ ]:
# clasificador_CART = DecisionTreeClassifier(
#     max_depth=5,
#     min_samples_split=5
# )
# clasificador_CART.fit(atr_train, obj_train)

# Hace lo mismo el best_estimator_

In [ ]:
mejor_CART = búsqueda_en_rejilla_CART.best_estimator_
mejor_CART.fit(atr_train, obj_train)
print("Accuracy en entrenamiento:", mejor_CART.score(atr_train, obj_train))
print("Accuracy en prueba:", mejor_CART.score(atr_test, obj_test))

Accuracy en entrenamiento: 0.7837078651685393
Accuracy en prueba: 0.7664104080425783


In [ ]:
predicciones_CART = mejor_CART.predict(atr_test)
print(classification_report(obj_test, predicciones_CART, target_names=[str(c) for c in label_enc.classes_]))

              precision    recall  f1-score   support

           0       0.80      0.68      0.74      1611
           1       0.74      0.84      0.79      1771

    accuracy                           0.77      3382
   macro avg       0.77      0.76      0.76      3382
weighted avg       0.77      0.77      0.76      3382



# Segundo conjunto de datos (nodos 20000..40000)
Este conjunto corresponde al subgrafo construido con los nodos identificados del 20000 al 40000 (rango fijo y reproducible).
Se usa como muestra manejable del grafo completo para extraer relaciones y combinar atributos.
El objetivo es reducir el volumen sin perder estructura, facilitando el analisis y el modelado.

## Primer modelo: Naive Bayes

Aplicaremos Naive Bayes como linea base porque es rapido y funciona bien con variables independientes o casi independientes. El modelo estima la probabilidad de cada clase dado un conjunto de caracteristicas y elige la clase con mayor probabilidad. Aunque la suposicion de independencia es fuerte, suele dar resultados competitivos y nos sirve para comparar con modelos mas complejos.

In [2]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

In [3]:
from sklearn.preprocessing import KBinsDiscretizer

# Cargar datos y preparar X/y
ruta = "generated_CSV/features_relaciones_20k_to_40k.csv"

df = pd.read_csv(ruta)
columnas_excluir = ["created_at", "update_at", "numeric_id"]
columnas_relacionales = ["degree_centrality", "closeness_centrality", "betweenness_centrality", "clustering", "pageRank"]

y = df["affiliate"].copy()
#DF 1: Todos los atributos excepto los de texto y la variable objetivo
X = df.drop(columns=columnas_excluir + ["affiliate"], errors="ignore").copy()
#DF 2: No atributos relacionales
#X = df.drop(columns=columnas_excluir + ["affiliate"] + columnas_relacionales, errors="ignore").copy()
#DF 3: Solo atributos relacionales
# X = df[columnas_relacionales].copy()
#DF 4: Todos los atributos originiales pero añadiendo solo pageRank y closeness_centrality
#X = df.drop(columns=columnas_excluir + ["affiliate"], errors="ignore").copy()
#X["pageRank"] = df["pageRank"]
# X["closeness_centrality"] = df["closeness_centrality"]

# Discretizar atributos continuos antes de codificar
atributos_continuos = ["degree_centrality", "closeness_centrality", "betweenness_centrality", "clustering", "pageRank"]

discretizador = KBinsDiscretizer(
    n_bins=40,
    encode="ordinal",
    strategy="kmeans",
)
X_disc = X.copy()
#Con el DF 2 hay que comentar esta parte porque no tenemos atributos continuos
X_disc[atributos_continuos] = discretizador.fit_transform(
    X_disc[atributos_continuos]
  )

# Codificar variables categoricas (incluye numericas como categorias si vienen como object)
encoder = OrdinalEncoder()
X_enc = encoder.fit_transform(X_disc)

# Codificar la variable objetivo
label_enc = LabelEncoder()
y_enc = label_enc.fit_transform(y)

c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 4 are removed. Consider decreasing the number of bins.
  warnings.warn(


In [4]:
# Particion train/test
atr_train, atr_test, obj_train, obj_test = train_test_split(
    X_enc, y_enc, test_size=0.2, random_state=94357, stratify=y_enc
)

In [5]:
tubería_NB = Pipeline([('preprocesador', discretizador),
                       ('naive_Bayes', CategoricalNB())])
rejilla_de_hiperparámetros = {
    'naive_Bayes__alpha': range(1, 10),
    'naive_Bayes__force_alpha': [True, False],
}


In [6]:
búsqueda_en_rejilla = GridSearchCV(tubería_NB,
                                   rejilla_de_hiperparámetros,
                                   scoring='accuracy',
                                   cv=10)
búsqueda_en_rejilla.fit(atr_train, obj_train)

c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (40). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 1 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (40). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in 

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...goricalNB())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'naive_Bayes__alpha': range(1, 10), 'naive_Bayes__force_alpha': [True, False]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",10
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the 

In [7]:
búsqueda_en_rejilla.best_estimator_

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocesador', ...), ('naive_Bayes', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"n_bins n_bins: int or array-like of shape (n_features,), default=5The number of bins to produce. Raises ValueError if ``n_bins < 2``.",40
,"encode encode: {'onehot', 'onehot-dense', 'ordinal'}, default='onehot'Method used to encode the transformed result.- 'onehot': Encode the transformed result with one-hot encoding and return a sparse matrix. Ignored features are always stacked to the right.- 'onehot-dense': Encode the transformed result with one-hot encoding and return a dense array. Ignored features are always stacked to the right.- 'ordinal': Return the bin identifier encoded as an integer value.",'ordinal'
,"strategy strategy: {'uniform', 'quantile', 'kmeans'}, default='quantile'Strategy used to define the widths of the bins.- 'uniform': All bins in each feature have identical widths.- 'quantile': All bins in each feature have the same number of points.- 'kmeans': Values in each bin have the same nearest center of a 1D k-means cluster.For an example of the different strategies see::ref:`sphx_glr_auto_examples_preprocessing_plot_discretization_strategies.py`.",'kmeans'
,"quantile_method quantile_method: {""inverted_cdf"", ""averaged_inverted_cdf"",""closest_observation"", ""interpolated_inverted_cdf"", ""hazen"",""weibull"", ""linear"", ""median_unbiased"", ""normal_unbiased""},default=""linear""Method to pass on to np.percentile calculation when usingstrategy=""quantile"". Only `averaged_inverted_cdf` and `inverted_cdf`support the use of `sample_weight != None` when subsampling is notactive... versionadded:: 1.7",'warn'
,"dtype dtype: {np.float32, np.float64}, default=NoneThe desired data-type for the output. If None, output dtype isconsistent with input dtype. Only np.float32 and np.float64 aresupported... versionadded:: 0.24",None
,"subsample subsample: int or None, default=200_000Maximum number of samples, used to fit the model, for computationalefficiency.`subsample=None` means that all the training samples are used whencomputing the quantiles that determine the binning thresholds.Since quantile computation relies on sorting each column of `X` andthat sorting has an `n log(n)` time complexity,it is recommended to u

In [ ]:
búsqueda_en_rejilla.best_score_

np.float64(0.742975411634867)

In [9]:
# Evaluacion
tubería_NB = Pipeline([('preprocesador', discretizador),
                       ('naive_Bayes', CategoricalNB(alpha=2, force_alpha=True))])
tubería_NB.fit(atr_train, obj_train)
tubería_NB.score(atr_test, obj_test)

c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (40). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 1 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (40). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\Payan\Desktop\Universidad\Trabajo-IA\TWITCH\.venv\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in 

0.7588817705299942

In [10]:
predicciones_NB = tubería_NB.predict(atr_test)
print(classification_report(obj_test, predicciones_NB, target_names=[str(c) for c in label_enc.classes_]))

              precision    recall  f1-score   support

           0       0.78      0.69      0.74      1665
           1       0.74      0.82      0.78      1769

    accuracy                           0.76      3434
   macro avg       0.76      0.76      0.76      3434
weighted avg       0.76      0.76      0.76      3434



## Segundo modelo: CART (Classification and Regression Trees)

CART es un algoritmo de aprendizaje supervisado que construye un arbol de decision mediante divisiones binarias recursivas. En cada nodo, el algoritmo selecciona el atributo y el umbral que mejor separan las clases, maximizando la pureza de los subconjuntos resultantes (usando metricas como el indice Gini o la entropia).

A diferencia de Naive Bayes, CART no asume independencia entre las caracteristicas y puede capturar relaciones no lineales e interacciones complejas entre variables. Ademas, es interpretable visualmente, ya que el arbol resultante muestra explicitamente las reglas de decision.

Para evitar el sobreajuste, aplicaremos poda del arbol limitando su profundidad maxima (`max_depth`) y el numero minimo de muestras necesarias para dividir un nodo (`min_samples_split`). Estos hiperparametros se optimizaran mediante busqueda en rejilla con validacion cruzada.

In [11]:
from sklearn.tree import DecisionTreeClassifier

In [12]:
clasificador_CART = DecisionTreeClassifier(random_state=94357)
rejilla_de_hiperparámetros_CART = {
    'max_depth': range(5, 50), # profundidad máxima del árbol para evitar el overfitting
    'min_samples_split': range(5, 25, 5) # número de muestras mínimo para dividir un nodo y prevenir el overfitting
}

In [13]:
búsqueda_en_rejilla_CART = GridSearchCV(clasificador_CART,
                                        rejilla_de_hiperparámetros_CART,
                                        scoring='accuracy',
                                        cv=10)
búsqueda_en_rejilla_CART.fit(atr_train, obj_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeC...m_state=94357)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': range(5, 50), 'min_samples_split': range(5, 25, 5)}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",10
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also 

In [14]:
búsqueda_en_rejilla_CART.best_params_

{'max_depth': 5, 'min_samples_split': 20}

In [15]:
búsqueda_en_rejilla_CART.best_score_

np.float64(0.769915165740614)

In [16]:
# clasificador_CART = DecisionTreeClassifier(
#     max_depth=5,
#     min_samples_split=5
# )
# clasificador_CART.fit(atr_train, obj_train)

# Hace lo mismo el best_estimator_

In [17]:
mejor_CART = búsqueda_en_rejilla_CART.best_estimator_
mejor_CART.fit(atr_train, obj_train)
print("Accuracy en entrenamiento:", mejor_CART.score(atr_train, obj_train))
print("Accuracy en prueba:", mejor_CART.score(atr_test, obj_test))

Accuracy en entrenamiento: 0.7775593417795252
Accuracy en prueba: 0.785381479324403


In [18]:
predicciones_CART = mejor_CART.predict(atr_test)
print(classification_report(obj_test, predicciones_CART, target_names=[str(c) for c in label_enc.classes_]))

              precision    recall  f1-score   support

           0       0.83      0.70      0.76      1665
           1       0.76      0.86      0.81      1769

    accuracy                           0.79      3434
   macro avg       0.79      0.78      0.78      3434
weighted avg       0.79      0.79      0.78      3434

